In [1]:
%pip install -q datasets pandas numpy scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
import re

from datasets import load_dataset

f:\Prabhleen New\practice\Machine Learning\OneAI\datasets\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import sys
!"{sys.executable}" -m pip install zstandard


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
from datasets import load_dataset

ds = load_dataset("agentlans/alpaca-prompts-annotated")
print(ds)

Generating test split: 100%|██████████| 10350/10350 [00:00<00:00, 202446.68 examples/s]

DatasetDict({
    train: Dataset({
        features: ['prompt', 'task_type_1', 'task_type_2', 'task_type_prob', 'creativity_scope', 'reasoning', 'contextual_knowledge', 'number_of_few_shots', 'domain_knowledge', 'no_label_reason', 'constraint_ct', 'prompt_complexity_score'],
        num_rows: 41397
    })
    test: Dataset({
        features: ['prompt', 'task_type_1', 'task_type_2', 'task_type_prob', 'creativity_scope', 'reasoning', 'contextual_knowledge', 'number_of_few_shots', 'domain_knowledge', 'no_label_reason', 'constraint_ct', 'prompt_complexity_score'],
        num_rows: 10350
    })
})


In [5]:
print(ds["train"].column_names)

['prompt', 'task_type_1', 'task_type_2', 'task_type_prob', 'creativity_scope', 'reasoning', 'contextual_knowledge', 'number_of_few_shots', 'domain_knowledge', 'no_label_reason', 'constraint_ct', 'prompt_complexity_score']


In [6]:
N = 5000

df = (
    ds["train"]
    .shuffle(seed=42)
    .select(range(N))
    .to_pandas()
)

print("Dataset shape:", df.shape)

Dataset shape: (5000, 12)


In [7]:
numeric_cols = [
    "prompt_complexity_score",
    "reasoning",
    "contextual_knowledge",
    "domain_knowledge",
    "constraint_ct"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

In [8]:
df["prompt"] = df["prompt"].astype(str).str.strip()

# Remove empty prompts
df = df[df["prompt"].str.len() > 5]

# Remove duplicates
df = df.drop_duplicates(subset=["prompt"])

In [9]:
def calculate_decomposition_score(row):

    prompt = row["prompt"].lower()

    score = 0

    complexity = row["prompt_complexity_score"]
    reasoning = row["reasoning"]
    context = row["contextual_knowledge"]
    domain = row["domain_knowledge"]
    constraint = row["constraint_ct"]

    # ----------------------------------------------
    # A. Complexity score
    # ----------------------------------------------

    if complexity >= 0.70:
        score += 3

    elif complexity >= 0.60:
        score += 2

    elif complexity >= 0.50:
        score += 1


    # ----------------------------------------------
    # B. Reasoning
    # ----------------------------------------------

    if reasoning >= 0.70:
        score += 2

    elif reasoning >= 0.55:
        score += 1


    # ----------------------------------------------
    # C. Constraints
    # ----------------------------------------------

    if constraint >= 0.70:
        score += 2

    elif constraint >= 0.50:
        score += 1


    # ----------------------------------------------
    # D. Contextual knowledge
    # ----------------------------------------------

    if context >= 0.75:
        score += 1


    # ----------------------------------------------
    # E. Domain knowledge
    # ----------------------------------------------

    if domain >= 0.65:
        score += 1


    # ----------------------------------------------
    # F. Explicit multi-step language
    # ----------------------------------------------

    sequence_patterns = [
        r"\bfirst\b",
        r"\bsecond\b",
        r"\bthird\b",
        r"\bthen\b",
        r"\bnext\b",
        r"\bafter that\b",
        r"\bfinally\b",
        r"\bstep 1\b",
        r"\bstep 2\b",
        r"\bstep 3\b",
        r"\bafterwards\b"
    ]

    sequence_count = sum(
        bool(re.search(pattern, prompt))
        for pattern in sequence_patterns
    )

    if sequence_count >= 2:
        score += 3

    elif sequence_count == 1:
        score += 1


    # ----------------------------------------------
    # G. Multiple task verbs
    # ----------------------------------------------

    task_verbs = [
        "research",
        "find",
        "search",
        "identify",
        "collect",
        "analyze",
        "analyse",
        "compare",
        "contrast",
        "evaluate",
        "calculate",
        "summarize",
        "summarise",
        "explain",
        "recommend",
        "suggest",
        "design",
        "create",
        "write",
        "implement",
        "build",
        "develop",
        "classify",
        "extract",
        "generate"
    ]

    detected_verbs = []

    for verb in task_verbs:
        if re.search(r"\b" + re.escape(verb) + r"\b", prompt):
            detected_verbs.append(verb)

    unique_verbs = len(set(detected_verbs))

    if unique_verbs >= 4:
        score += 3

    elif unique_verbs >= 3:
        score += 2

    elif unique_verbs >= 2:
        score += 1


    # ----------------------------------------------
    # H. Multiple output requirements
    # ----------------------------------------------

    output_patterns = [
        "provide a list",
        "give a list",
        "create a table",
        "make a table",
        "write a report",
        "provide recommendations",
        "give recommendations",
        "provide examples",
        "include examples",
        "explain and",
        "compare and",
        "analyze and",
        "research and"
    ]

    output_count = sum(
        pattern in prompt
        for pattern in output_patterns
    )

    if output_count >= 2:
        score += 2

    elif output_count == 1:
        score += 1


    # ----------------------------------------------
    # I. Explicit conjunctions connecting tasks
    # ----------------------------------------------

    conjunction_patterns = [
        r"\band then\b",
        r"\band also\b",
        r"\bas well as\b",
        r"\bin addition\b",
        r"\bafter analyzing\b",
        r"\bbased on your analysis\b",
        r"\bbased on the results\b",
        r"\bfinally recommend\b",
        r"\bthen recommend\b"
    ]

    conjunction_count = sum(
        bool(re.search(pattern, prompt))
        for pattern in conjunction_patterns
    )

    score += min(conjunction_count * 2, 4)


    # ----------------------------------------------
    # J. Strong multi-stage combinations
    # ----------------------------------------------

    strong_combinations = [
        ("research", "compare"),
        ("research", "recommend"),
        ("research", "analyze"),
        ("find", "compare"),
        ("find", "evaluate"),
        ("analyze", "recommend"),
        ("analyze", "suggest"),
        ("compare", "recommend"),
        ("collect", "analyze"),
        ("collect", "evaluate"),
        ("calculate", "recommend"),
        ("analyze", "create"),
        ("research", "summarize"),
        ("identify", "evaluate")
    ]

    for word1, word2 in strong_combinations:

        if (
            re.search(r"\b" + word1 + r"\b", prompt)
            and
            re.search(r"\b" + word2 + r"\b", prompt)
        ):
            score += 3


    return score


df["decomposition_score"] = df.apply(
    calculate_decomposition_score,
    axis=1
)

In [10]:
print(
    df["decomposition_score"]
    .value_counts()
    .sort_index()
)

decomposition_score
0      761
1     1172
2     1231
3      669
4      575
5      347
6      161
7       55
8       18
9        5
10       5
11       1
Name: count, dtype: int64


In [11]:
df = df.sort_values(
    "decomposition_score",
    ascending=False
).reset_index(drop=True)

In [12]:
positive_candidates = df[
    df["decomposition_score"] >= 7
].copy()

# Very low scoring prompts
negative_candidates = df[
    df["decomposition_score"] <= 3
].copy()


print("Positive candidates:", len(positive_candidates))
print("Negative candidates:", len(negative_candidates))

Positive candidates: 84
Negative candidates: 3833


In [13]:
if len(positive_candidates) < 3000:

    threshold = df["decomposition_score"].quantile(0.60)

    positive_candidates = df[
        df["decomposition_score"] >= threshold
    ].copy()


if len(negative_candidates) < 3000:

    threshold = df["decomposition_score"].quantile(0.40)

    negative_candidates = df[
        df["decomposition_score"] <= threshold
    ].copy()

In [14]:
positive_candidates = positive_candidates[
    ~positive_candidates["prompt"].isin(
        negative_candidates["prompt"]
    )
]


In [15]:
np.random.seed(42)

positive = positive_candidates.sample(
    n=3000,
    random_state=42
).copy()

negative = negative_candidates.sample(
    n=3000,
    random_state=42
).copy()

ValueError: Cannot take a larger sample than population when 'replace=False'

In [16]:
# Sort by decomposition score
df = df.sort_values(
    "decomposition_score",
    ascending=False
).reset_index(drop=True)

# Highest 3000 → label 1
positive = df.iloc[:3000].copy()
positive["label"] = 1

# Lowest 3000 → label 0
negative = df.iloc[-3000:].copy()
negative["label"] = 0

# Combine
final_df = pd.concat(
    [positive, negative],
    ignore_index=True
)

# Shuffle
final_df = final_df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

# Final columns
final_df = final_df[["prompt", "label"]]

# Save
final_df.to_csv(
    "prompt_decomposition_dataset_6000.csv",
    index=False
)

print(final_df.shape)
print(final_df["label"].value_counts())

(6000, 2)
label
1    3000
0    3000
Name: count, dtype: int64


In [17]:
positive["label"] = 1
negative["label"] = 0

In [18]:
final_df = pd.concat(
    [positive, negative],
    ignore_index=True
)


In [19]:
final_df = final_df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

In [20]:
final_df = final_df[
    ["prompt", "label"]
]

In [21]:
output_file = "prompt_decomposition_dataset_6000.csv"

final_df.to_csv(
    output_file,
    index=False
)